# Notebook para la predicción de una métrica cuantitativa en función de los modelos generalizados previamente

## Importación de librerías necesarias

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path
src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

In [3]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('predict_flow', memory_tuning=True)
spark = spark_utils.spark

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-64d3af71-8f51-4aa1-9196-4e25bc1ec1ed;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 124ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [4]:
from pyspark.sql import functions as F, types as T

## Importar información de referencia

In [5]:
meta_items = spark.read.format('delta').load(spark_utils.path('meta_items', 'bronze'))

In [6]:
ASIN = "1610121147"

## Recibir entrada de usuario separada por componente

In [7]:
meta_items.filter(F.col('parent_asin') == 'B000NNS5XS').limit(1).toPandas().values

array([['Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black) (OLD MODEL)',
        'Camera & Photo',
        list(['7.1-megapixel CCD captures enough detail for photo-quality 15 x 20-inch prints', '3x optical zoom; ISO 1600 and High ISO Auto', 'DIGIC III Image Processor; Face Detection AF/AE', 'Selectable shooting modes and special scene modes', 'Print/Share button makes direct printing simple']),
        list(['Product Description', 'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black)', 'From the Manufacturer', 'Canon looked to the very first Elph for inspiration when designing the PowerShot SD1000 Digital Elph, and came up with a quintessential iteration of the icon: slim, clean-lined and fully flat. Inside, the SD1000 Digital Elph looks only to the future: 7.1 megapixels, a 3x optical zoom and advanced DIGIC III ensure top-quality images, while focus is fast and sharp and red-eye is automatically corrected. The large and more color

In [8]:
TITLE = 'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black) (OLD MODEL)'
DESCRIPTION = ['Product Description', 'Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black)', 'From the Manufacturer', 'Canon looked to the very first Elph for inspiration when designing the PowerShot SD1000 Digital Elph, and came up with a quintessential iteration of the icon: slim, clean-lined and fully flat. Inside, the SD1000 Digital Elph looks only to the future: 7.1 megapixels, a 3x optical zoom and advanced DIGIC III ensure top-quality images, while focus is fast and sharp and red-eye is automatically corrected. The large and more colorful LCD screen now has a tough, anti-reflective coating that makes it as durable as it is beautiful.', 'PowerShot SD1000 Highlights', '1x zoom/3x zoom', 'Slim, stylish 7.1-megapixel digital Elph with 3x optical zoom', "Great design is just part of the PowerShot SD1000 Digital Elph story. Inside is all the power you need to capture the moments of your life - beautifully.   The 7.1-megapixel CCD records a wealth of detail - enough to let you enlarge and crop at will. Images are rich and sharp with lifelike depth. The camera's Genuine Canon 3x optical zoom not only gets you in close, but performs with all the clarity and brilliance you'd expect from the world's leader in advanced optics technology.", 'DIGIC III image processor with improved Face Detection and Red-eye Correction', "With DIGIC III, you get images of superior quality, the camera functions at top efficiency and battery life is significantly enhanced. What's more, DIGIC III enables Canon's newly improved Face Detection Technology and Red-eye Correction to give you better, more true-to-life people shots. Simply press the Shutter Button halfway down, and the PowerShot SD1000 Digital Elph automatically pinpoints the faces in the scene and chooses the ideal focus point. To keep every face looking bright and natural - without scary red eyes - the camera controls exposure settings and flash, so every shot is just what you were shooting for.", 'Face Detection AF/AE', "finds multiple faces in the frame and sets the most suitable focus point, when the shutter button is pressed halfway. And an additional feature, Face Detection FE adjusts the flash, based on a person's face on the screen. Exposure and flash are controlled to ensure proper illumination of both the faces and the overall scene, eliminating the common problem of darkened or overexposed faces.", 'Face Detection in action', 'Red-eye Correction', 'detects and automatically corrects red-eye during playback for both regular and flash photography. In unusual cases where red-eye is not automatically detected, it can easily be corrected manually during playback mode from the LCD screen.', 'iSAPS Technology', 'is an entirely original scene-recognition technology developed for digital cameras by Canon. Using an internal database of thousands of different photos, iSAPS works with the fast DIGIC III Image Processor to improve focus speed and accuracy, as well as exposure and white balance.', 'Vivid, high-resolution 2.5-inch PureColor LCD', "The camera's 2.5-inch LCD screen gives you the big picture, whether you're shooting, reviewing or showing off your images. This extra-durable, high-resolution screen with tough scratch-resistant coating on the anti-reflective, PureColor LCD screen offers a crisp, clear picture to make shooting, playback and using the camera's menu functions especially convenient. Clear and bright, it also features Night Display for easy viewing in low light.", 'ISO 1600 and High ISO Auto to reduce image blur and expand low-light shooting capability', 'The PowerShot SD1000 Digital Elph features new ISO 1600 and High ISO Auto settings that reduce the effects of camera shake and sharpen subjects in low-light situations, giving you greater flexibility for shooting.', 'Five movie modes including 30 fps VGA, Time Lapse and Fast Frame Rate', "With a highly flexible movie mode, you can create the movie that's perfect for any application. Select from VGA (640 x 480 pixels) and QVGA (320 x 240 pixels), with frame rates of 30 fps and 15 fps for recording up to 1 hour or 4GB. Also choose from Fast Frame Rate (QVGA; 320 x 240 pixels) recording at 60 fps for up to 1 minute, Compact Movie Mode (QQVGA; 160 x 120 pixels) recording at 15 fps for up to 3 minutes, and Time Lapse (640 x 480) recording at 1 or 2 sec. intervals. The PowerShot SD1000 Digital Elph supports the USB 2.0 Hi-Speed standard, so you'll enjoy the fastest possible data transfer speeds when using a USB 2.0 Hi-Speed compatible computer.", 'Print/Share button for easy direct printing and downloading', "The PowerShot SD1000 Digital Elph's Print/Share button makes direct printing easier than ever. Simply connect the SD1000 Digital Elph to a Canon CP, Selphy or Pixma photo printer or any PictBridge compatible photo printer, press the lighted Print/Share button and print! Also use the Print/Share button to transfer images to a computer (Windows and Macintosh).   Print your own ID photos in 28 different sizes or use the Movie Print function to output multiple stills from a recorded movie on a single sheet with a Canon Selphy compact photo printer.", 'Direct photo printers', 'For desktop large-format printing, try one of the direct photo printers that allow you to print without a computer in one of two ways: plug your compatible PowerShot camera into the direct photo printer using the supplied USB interface cable, or simply insert a memory card into the supplied adapter. You can also connect the printer to your computer for more options. Print high-resolution, borderless images as postcards or 8.5 x 11-inch sheets within minutes.', 'Compact photo printers', "Compact photo printers let you produce versatile, fun 4 x 6-inch postcards, 4 x 8-inch wide greeting cards or credit-card size prints in just two easy steps: connect and press print. Control the printer right from your camera's LCD screen. You get durable, dye-sublimated prints quickly with or without borders. Assorted paper types let you create mini or credit card size labels. You can even take select compact photo printers to a party or an outdoor picnic using an optional rechargeable battery."]
FEATURES = [
   '7.1-megapixel CCD captures enough detail for photo-quality 15 x 20-inch prints', '3x optical zoom; ISO 1600 and High ISO Auto', 'DIGIC III Image Processor; Face Detection AF/AE', 'Selectable shooting modes and special scene modes', 'Print/Share button makes direct printing simple'
]

In [9]:
meta_items_title_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_title_text_clean',
    catalog = 'silver.preprocess'
))

In [10]:
import pyspark.sql.functions as F
meta_items_title_text_clean.filter(F.col('parent_asin') == ASIN).limit(1).toPandas()

,parent_asin,title
0,1610121147,NEWEST Black Color Arachnophobia Durable Alumi...


In [11]:
meta_items_title_text_clean.select('parent_asin').distinct().count()

3125022

In [12]:
from src.utils.spark import SparkUtils
from src.utils.preprocessors.clean_words import CleanWords
from src.utils.models.clustering.LSHNeighborsClustering import LSHNeighborsClustering
from src.utils.models.encoding.SummarizeEncoding import SummarizeEncoding
from src.utils.models.encoding.SentenceEncoder import SentenceEncoder
from src.gold.training.pca import PCAEncoder

In [13]:
from src.utils.testers.FullTester import FullTester
full_tester = FullTester(
    spark=spark,
    spark_utils=spark_utils,
    use_mini_llm=False
)

full_tester.set_components({
    "title": TITLE,
    "description": DESCRIPTION,
    "features": FEATURES,
})

I0000 00:00:1763951358.064107  574539 service.cc:146] XLA service 0x1858e730 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763951358.064133  574539 service.cc:154]   StreamExecutor device (0): Host, Default Version


21:29:38.455 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


INFO:FullTester:Review predictor preparer initialized successfully              
INFO:FullTester:Starting to initialize binary model


/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
INFO:FullTester:Starting to initialize categorical model


INFO:FullTester:Starting to initialize category predictor


/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 24 variables whereas the saved optimizer has 46 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
INFO:FullTester:Starting to initialize binary model category


/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 30 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
INFO:FullTester:Starting to initialize predictions by model
INFO:FullTester:FullTester initialization completed
INFO:FullTester:Setting components
INFO:FullTester:Components set successfully


In [14]:
full_tester.clean_components()
full_tester.separate_sentences_per_component()
full_tester.encode_sentences()
full_tester.summarize_sentences_by_component()
full_tester.pca_encode()
full_tester.find_pairs()

INFO:FullTester:Starting to clean components


21:29:58.814 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


INFO:FullTester:Components cleaned successfully                                 
INFO:FullTester:Starting to separate sentences per component


21:30:04.366 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


21:30:11.078 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


...

INFO:FullTester:PCA encoding completed successfully                             
INFO:FullTester:Starting to find similar pairs


[INFO] Euclidean threshold: 0.44721359549995787


21:31:37.124 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


INFO:FullTester:Similar pairs found successfully                                


DataFrame[entity_id: string, features: vector, cosine_sim: double]

In [15]:
full_tester.build_models_inputs()

INFO:FullTester:Starting to build models inputs


21:41:21.096 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


21:41:33.102 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


21:41:38.114 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null


INFO:FullTester:Models inputs built successfully                                


(DataFrame[combined_feat_0: double, combined_feat_1: double, combined_feat_2: double, combined_feat_3: double, combined_feat_4: double, combined_feat_5: double, combined_feat_6: double, combined_feat_7: double, combined_feat_8: double, combined_feat_9: double, combined_feat_10: double, combined_feat_11: double, combined_feat_12: double, combined_feat_13: double, combined_feat_14: double, combined_feat_15: double, combined_feat_16: double, combined_feat_17: double, combined_feat_18: double, combined_feat_19: double, combined_feat_20: double, combined_feat_21: double, combined_feat_22: double, combined_feat_23: double, combined_feat_24: double, combined_feat_25: double, combined_feat_26: double, combined_feat_27: double, combined_feat_28: double, combined_feat_29: double, combined_feat_30: double, combined_feat_31: double, combined_feat_32: double, combined_feat_33: double, combined_feat_34: double, combined_feat_35: double, combined_feat_36: double, combined_feat_37: double, combined_fe

In [16]:
full_tester.generate_prediction()

INFO:FullTester:Starting to generate predictions
INFO:FullTester:Generating prediction for binary model


29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


INFO:FullTester:Generating prediction for categorical model


29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


INFO:FullTester:Generating prediction for category prediction model


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step


INFO:FullTester:Preparing binary model inputs with categorical model predictions
INFO:FullTester:Generating prediction for binary model category                 


29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


INFO:FullTester:Predictions generated successfully


{'binary_model': 0.8068562201387477,
 'categorical_model': None,
 'category_prediction_model': array([3.63650804e-10, 1.84526164e-06, 1.34538734e-12, 7.89929633e-09,
        6.01214989e-09, 1.25348185e-08, 5.80531964e-17, 4.94976499e-12,
        9.99998093e-01], dtype=float32),
 'binary_model_category': 0.8023597500730119,
 'unsupervised': None,
 'category_model': array([6.94619826e-10, 6.22258636e-02, 4.45978653e-02, 9.13630675e-02,
        2.23614420e-01, 5.78198773e-01])}

In [17]:
full_tester.predictions_by_model["category_prediction_model"][0]

3.636508e-10

In [18]:
full_tester.generate_prediction_jtbd()

INFO:FullTester:Starting to generate JTBD prediction


In [19]:
full_tester.predictions_by_model["unsupervised"]

0.8715119218104896

In [20]:
full_tester.predictions_by_model

{'binary_model': 0.8068562201387477,
 'categorical_model': None,
 'category_prediction_model': array([3.63650804e-10, 1.84526164e-06, 1.34538734e-12, 7.89929633e-09,
        6.01214989e-09, 1.25348185e-08, 5.80531964e-17, 4.94976499e-12,
        9.99998093e-01], dtype=float32),
 'binary_model_category': 0.8023597500730119,
 'unsupervised': 0.8715119218104896,
 'category_model': array([6.94619826e-10, 6.22258636e-02, 4.45978653e-02, 9.13630675e-02,
        2.23614420e-01, 5.78198773e-01])}

In [21]:
meta_items.filter(F.col('title') == F.lit("Canon PowerShot SD1000 7.1MP Digital Elph Camera with 3x Optical Zoom (Black) (OLD MODEL)")).show()

+--------------------+--------------+--------------------+--------------------+--------------+-------------+------+-----+-----------+--------------------+--------------------+--------------------+
|               title| main_category|            features|         description|average_rating|rating_number| price|store|parent_asin|          categories|             details|              images|
+--------------------+--------------+--------------------+--------------------+--------------+-------------+------+-----+-----------+--------------------+--------------------+--------------------+
|Canon PowerShot S...|Camera & Photo|[7.1-megapixel CC...|[Product Descript...|           4.1|          966|249.95|Canon| B000NNS5XS|[Electronics, Cam...|{"Auto Focus Tech...|[{"thumb":"https:...|
+--------------------+--------------+--------------------+--------------------+--------------+-------------+------+-----+-----------+--------------------+--------------------+--------------------+



In [22]:
meta_items.filter(F.col('parent_asin') == 'B074JLV5XX').limit(1).toPandas().values

array([['VESA Mount Adapter for Dell S2218, S2318, S2319, S2418, S2419, S2718, S2719 Monitors | Does Not Fit Ultrathin Monitors S2718D, S2719HM, and S2719DM | [Patented] - by HumanCentric',
        'Office Products',
        list(['FITS SELECT DELL S-SERIES MONITORS: This bracket is tested and guaranteed to work with: Dell S2218H, S2318HX, S2318HN, S2318NX, S2319H, S2319HN, S2418HX, S2418HN, S2418NX, S2419H, S2419HN, S2718HX, S2718HN, S2718NX, S2719H, and S2719HN (Does not fit any Ultrathin monitors such as S2718D, S2419HM and S2719DM)', 'IMPORTANT NOTE: Does not fit the SE2219H, SE2419H, and SE2719H - Please select the size labeled "Dell S2*18, S2*19, and SE2*19 (Not Ultrathin Models)" for the VESA adapter for those model numbers', 'MOUNT YOUR DELL MONITOR ON A STANDARD VESA MOUNT - Even though these monitors weren’t made with mounting holes, our convenient bracket enables you to connect your Dell monitor anyway! Whether you’re looking to mount it on the wall or just get it off your d

In [23]:
meta_items.columns

['title',
 'main_category',
 'features',
 'description',
 'average_rating',
 'rating_number',
 'price',
 'store',
 'parent_asin',
 'categories',
 'details',
 'images']

In [24]:
meta_items.filter(F.col('parent_asin') == 'B000Q30420').limit(1).toPandas().values

array([['Canon PowerShot SD850 IS 8.0 MP Digital Elph Camera with 4x Optical Image Stabilized Zoom (OLD MODEL)',
        'Camera & Photo',
        list(['8.0-megapixel CCD captures enough detail for photo-quality 16 x 22-inch prints', '4x Optical Image Stabilized zoom for steady, long zoom shooting', 'High-resolution 2.5-inch PureColor LCD with scratch-resistant, anti-reflection coating', 'Sensitivity range to ISO 1600 for sharper photos in low light', 'Print/Share Button for easy direct printing and downloading']),
        list(['Product Description', 'Discover a new inspiration. The PowerShot SD850 IS Digital Elph is a digital camera that will really get your creative juices flowing. It starts with a high-resolution 8-megapixel CCD, a 4x optical zoom with Canon’s exclusive UA Lens and an Optical Image Stabilizer for steady zooming. There’s also a DIGIC III Image Processor with Face Detection and red-eye correction, an ISO 1600 setting for sharper images in low light, 5 Movie Modes an

In [25]:
meta_items.filter(F.col('parent_asin') == 'B000Q30420').limit(1).toPandas().values[0][3]

['Product Description',
 'Discover a new inspiration. The PowerShot SD850 IS Digital Elph is a digital camera that will really get your creative juices flowing. It starts with a high-resolution 8-megapixel CCD, a 4x optical zoom with Canon’s exclusive UA Lens and an Optical Image Stabilizer for steady zooming. There’s also a DIGIC III Image Processor with Face Detection and red-eye correction, an ISO 1600 setting for sharper images in low light, 5 Movie Modes and a 2.5-inch PureColor LCD with scratch-resistant, anti-reflective coating for easy viewing.',
 'From the Manufacturer',
 'Manufacturer Description',
 'Discover a new inspiration. The PowerShot SD850 IS Digital Elph is a digital camera that will really get your creative juices flowing. It starts with a high-resolution 8-megapixel CCD, a 4x optical zoom with Canon’s exclusive UA Lens and an Optical Image Stabilizer for steady zooming. There’s also a DIGIC III Image Processor with Face Detection and red-eye correction, an ISO 1600